# Section 4 — Outlier Detection with Tukey's Rule and Z-score (25 pts)

## Learning Objectives
- Detect outliers using Tukey's rule (IQR) and Z-score
- Compare both methods with boxplots and summary tables
- Interpret extremes in the context of air pollution data

## Your Decisions Log

| Step | Decision | Evidence (plot / table / stat) |
|------|----------|--------------------------------|
|      |          |                                |

## Suggested Variables
CO(GT), C6H6(GT), NOx(GT), NO2(GT), T, RH, AH

## Tasks
- a) Generate boxplots for selected variables
- b) Detect outliers using Tukey's rule
- c) Detect outliers using Z-score
- d) Create outlier count/percentage table per method
- e) Compare results of both methods

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.config import RAW_DATA_PATH, MISSING_SENTINEL, KNN_K_VALUES, ZSCORE_THRESHOLD, FIGURES_DIR
from src.load_data import load_raw_air_quality, build_datetime
from src.missing_values import replace_sentinel_with_nan
from src.knn_imputation import select_numerical_columns, knn_impute
from src.outliers import (
    tukey_bounds,
    detect_tukey_outliers,
    detect_zscore_outliers,
    outlier_report,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Rebuild the imputed dataset so the outlier notebook can run independently.
df = load_raw_air_quality(RAW_DATA_PATH)
df = build_datetime(df)
df = replace_sentinel_with_nan(df, sentinel=MISSING_SENTINEL)
num_cols = select_numerical_columns(df)
df = knn_impute(df, num_cols, n_neighbors=KNN_K_VALUES[-1])

# Use the variables that are most interpretable for pollution analysis.
OUTLIER_COLUMNS = ['CO(GT)', 'C6H6(GT)', 'T', 'RH', 'AH']
display(df[OUTLIER_COLUMNS].head())

In [ ]:
# Boxplots give a fast view of spread and extreme values before formal detection.
fig, axes = plt.subplots(1, len(OUTLIER_COLUMNS), figsize=(4 * len(OUTLIER_COLUMNS), 5))
if len(OUTLIER_COLUMNS) == 1:
    axes = [axes]

for axis, column in zip(axes, OUTLIER_COLUMNS):
    sns.boxplot(y=df[column], ax=axis, color='#4c72b0')
    axis.set_title(column)
    axis.set_xlabel('')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'outlier_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detect outliers with both rules and keep the counts visible for each variable.
outlier_details = []
for column in OUTLIER_COLUMNS:
    tukey_mask = detect_tukey_outliers(df[column])
    zscore_mask = detect_zscore_outliers(df[column], threshold=ZSCORE_THRESHOLD)
    lower, upper = tukey_bounds(df[column])
    outlier_details.append({
        'variable': column,
        'tukey_count': int(tukey_mask.sum()),
        'zscore_count': int(zscore_mask.sum()),
        'tukey_lower': lower,
        'tukey_upper': upper,
    })

outlier_df = pd.DataFrame(outlier_details)
display(outlier_df)

In [ ]:
# Summarize the outliers in one table so the decision is easy to audit.
report = outlier_report(df, OUTLIER_COLUMNS)
display(report)

## Guiding Questions

1. **Do Tukey's rule and Z-score detect the same observations as outliers?**

   No. They overlap in some cases, but they do not always flag the same observations. Z-score works better when the data are close to normal, while Tukey's rule is more robust on skewed data.

2. **Are the extreme values likely to be measurement errors, sensor behavior, or real pollution events?**

   It is likely a mix. Some extreme values may come from sensor glitches or invalid readings, while others may reflect real pollution peaks or periods of unstable sensor behavior. Each case should be checked in context.

3. **Which method is more appropriate for skewed variables? Explain your reasoning.**

   Tukey's rule is more appropriate for skewed variables because it relies on quartiles instead of the mean and standard deviation, so it is less sensitive to asymmetry and extreme tails.